# CropShield AI — Train a free plant disease classifier

This notebook trains a 5-class MobileNetV2 image classifier using the public PlantVillage dataset. It is intended for a student demonstration and close-up leaf photos. It does **not** validate drone-height aerial images or claim support for every Andhra Pradesh crop.

Classes: **Healthy, Leaf Blight, Mosaic Virus, Rust, Powdery Mildew**. Selected source classes: maize healthy / northern leaf blight / common rust; tomato healthy / tomato mosaic virus; squash healthy / powdery mildew.

Run cells from top to bottom. A free GPU is optional, not guaranteed. No paid API is used.

In [ ]:
!pip -q install tensorflow-datasets

In [ ]:
import json, os, numpy as np, tensorflow as tf, tensorflow_datasets as tfds
from google.colab import files

SEED = 42
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE
TARGET_NAMES = ["Healthy", "Leaf Blight", "Mosaic Virus", "Rust", "Powdery Mildew"]
print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

In [ ]:
builder = tfds.builder("plant_village")
builder.download_and_prepare()
label_names = builder.info.features["label"].names
print("PlantVillage source categories:", len(label_names))
for i, name in enumerate(label_names):
    if any(term in name.lower() for term in ["healthy", "northern_leaf_blight", "common_rust", "tomato_mosaic_virus", "powdery_mildew"]):
        print(i, name)

In [ ]:
# Match exact public-dataset category names. Stop instead of silently substituting a different disease.
required = {
    "Corn_(maize)___healthy": "Healthy",
    "Tomato___healthy": "Healthy",
    "Squash___healthy": "Healthy",
    "Corn_(maize)___Northern_Leaf_Blight": "Leaf Blight",
    "Tomato___Tomato_mosaic_virus": "Mosaic Virus",
    "Corn_(maize)___Common_rust_": "Rust",
    "Squash___Powdery_mildew": "Powdery Mildew",
}
label_to_id = {name: i for i, name in enumerate(label_names)}
missing = [name for name in required if name not in label_to_id]
if missing:
    raise ValueError("Expected PlantVillage labels not found: " + repr(missing) + ". Inspect the printed label list and update mapping carefully.")
source_mapping = {label_to_id[source]: TARGET_NAMES.index(target) for source, target in required.items()}
print("Using source categories:")
for source, target in required.items(): print(f"  {source} -> {target}")

In [ ]:
raw = tfds.load("plant_village", split="train", as_supervised=True)

def prepare_one(image, target_id):
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.keras.applications.mobilenet_v2.preprocess_input(tf.cast(image, tf.float32))
    return image, tf.cast(target_id, tf.int32)

def make_class_splits(source_id, target_id, cap=1400):
    class_ds = raw.filter(lambda image, label: tf.equal(label, source_id))
    class_ds = class_ds.shuffle(2500, seed=SEED, reshuffle_each_iteration=False)
    count = sum(1 for _ in class_ds)
    count = min(count, cap)
    if count < 50:
        raise ValueError(f"Too few examples ({count}) for source label {label_names[source_id]}")
    class_ds = class_ds.take(count)
    n_train = int(count * 0.8)
    n_val = int(count * 0.1)
    train = class_ds.take(n_train).map(lambda x, y: prepare_one(x, target_id), num_parallel_calls=AUTOTUNE)
    rest = class_ds.skip(n_train)
    val = rest.take(n_val).map(lambda x, y: prepare_one(x, target_id), num_parallel_calls=AUTOTUNE)
    test = rest.skip(n_val).map(lambda x, y: prepare_one(x, target_id), num_parallel_calls=AUTOTUNE)
    print(label_names[source_id], "->", TARGET_NAMES[target_id], "images:", count)
    return train, val, test

train_parts, val_parts, test_parts = [], [], []
for source_id, target_id in source_mapping.items():
    tr, va, te = make_class_splits(source_id, target_id)
    train_parts.append(tr); val_parts.append(va); test_parts.append(te)
train_ds = train_parts[0]
val_ds = val_parts[0]
test_ds = test_parts[0]
for part in train_parts[1:]: train_ds = train_ds.concatenate(part)
for part in val_parts[1:]: val_ds = val_ds.concatenate(part)
for part in test_parts[1:]: test_ds = test_ds.concatenate(part)
train_ds = train_ds.shuffle(3000, seed=SEED).batch(BATCH_SIZE).prefetch(AUTOTUNE)
val_ds = val_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
test_ds = test_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
print("Datasets ready")

In [ ]:
base = tf.keras.applications.MobileNetV2(input_shape=(224,224,3), include_top=False, weights="imagenet")
base.trainable = False
inputs = tf.keras.Input(shape=(224,224,3))
x = base(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.25)(x)
outputs = tf.keras.layers.Dense(len(TARGET_NAMES), activation="softmax")(x)
model = tf.keras.Model(inputs, outputs)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", patience=2, factor=0.3),
]
history = model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=callbacks)

In [ ]:
print("Held-out test results:")
test_loss, test_accuracy = model.evaluate(test_ds)
print(f"Test accuracy: {test_accuracy:.2%} (on this dataset split; not a guarantee for real field/drone images)")

In [ ]:
# Save the exact model and label order expected by the website.
model_path = "/content/plant_disease_model.keras"
labels_path = "/content/class_names.json"
model.save(model_path)
with open(labels_path, "w", encoding="utf-8") as f:
    json.dump(TARGET_NAMES, f, indent=2)
print("Saved:", model_path, labels_path)
files.download(model_path)
files.download(labels_path)

## Install in the website

Copy both downloaded files into the extracted project's `models` folder. Keep the filenames exactly `plant_disease_model.keras` and `class_names.json`. Then run `run.bat`.

**Limitations:** PlantVillage is mainly controlled leaf imagery. This model's disease classes are maize, tomato, and squash examples; it does not establish disease recognition for rice, chilli, cotton, or aerial whole-field photos. Collect and test representative local field images before presenting it as a reliable drone system.